## 1. Database Overview

This notebook builds a relational SQLite3 database from the cleaned Data Analyst job datasets prepared in Notebooks 01 and 02.

The project examines whether Data Analyst positions that require a degree differ from positions that do not require a degree in areas such as salary, employment type, remote work, location, and technical skill requirements.

The cleaned datasets are stored in two database tables:

* `degree_jobs` — Data Analyst job postings identified as requiring a degree.
* `no_degree_jobs` — Data Analyst job postings identified as not requiring a degree.

SQLite3 is used to store the cleaned data and perform SQL-based analysis. Separating the two populations into their own tables preserves the distinction that is central to the project's research question while allowing the datasets to be compared using SQL.

The database portion of the project demonstrates how cleaned Python/Pandas data can be transferred into a relational database and analyzed using SQL.


## 2. Load Cleaned Datasets

The cleaned Data Analyst job datasets generated in Notebooks 01 and 02 are loaded into this notebook for database creation and SQL analysis.

The two datasets represent separate populations:

* **Degree-required jobs:** Data Analyst positions identified as requiring a degree.
* **No-degree jobs:** Data Analyst positions identified as not requiring a degree.

The cleaned CSV files are loaded using relative paths so that the project remains portable across operating systems and can be reproduced from the repository.

The datasets will be stored as separate tables in the SQLite3 database and will be validated before the database is created.


In [ ]:
# Load cleaned datasets
import pandas as pd

degree_jobs = pd.read_csv("../data/degree_jobs_cleaned.csv")
no_degree_jobs = pd.read_csv("../data/no_degree_jobs_cleaned.csv")

## 3. Validate Dataset Compatibility

The cleaned degree-required and no-degree datasets were designed with a consistent analytical structure so they can be stored and compared within the same SQLite database.

Both datasets contain the same core job, salary, location, employment, date, and technical-skill fields. The primary difference is the population-specific field used to describe the degree requirement:

* `degree_type` is a string field in the `degree_jobs` table that identifies the type of degree requirement.
* `no_degree_mention` is a Boolean field in the `no_degree_jobs` table that identifies whether a no-degree designation was present.

Both datasets also contain `degree_flag`, which provides a standardized indicator for distinguishing the two populations.

The consistent structure allows the two tables to be analyzed and compared using SQL while preserving the information specific to each population.


In [6]:
degree_jobs.columns.tolist()
no_degree_jobs.columns.tolist()

['job_id',
 'job_title',
 'salary_usd',
 'company_location',
 'is_remote',
 'employee_location',
 'job_skills',
 'no_degree_mention',
 'posting_date',
 'company_name',
 'has_python',
 'has_sql',
 'has_tableau',
 'degree_flag',
 'salary_tier',
 'employment_type']

In [8]:
print("Degree jobs columns:")
print(degree_jobs.columns.tolist())

Degree jobs columns:
['job_id', 'job_title', 'salary_usd', 'employment_type', 'company_location', 'is_remote', 'employee_location', 'job_skills', 'degree_type', 'posting_date', 'company_name', 'has_python', 'has_sql', 'has_tableau', 'degree_flag', 'salary_tier']


In [10]:
print("No-degree jobs columns:")
print(no_degree_jobs.columns.tolist())

No-degree jobs columns:
['job_id', 'job_title', 'salary_usd', 'company_location', 'is_remote', 'employee_location', 'job_skills', 'no_degree_mention', 'posting_date', 'company_name', 'has_python', 'has_sql', 'has_tableau', 'degree_flag', 'salary_tier', 'employment_type']


In [2]:
degree_jobs.shape

(759, 16)

In [3]:
no_degree_jobs.shape

(18294, 16)

In [7]:
degree_jobs['job_id'].is_unique, no_degree_jobs['job_id'].is_unique

(True, True)

## 4. Database Design

### Why Two Tables?

The database uses two tables to represent the two populations at the center of this project: Data Analyst jobs that require a degree and Data Analyst jobs that do not require a degree.

Keeping these populations in separate tables preserves the distinction established during the data-cleaning process while allowing their characteristics to be analyzed and compared using SQL.

The two tables share the same core analytical structure, including job information, salary information, company and location data, employment information, remote-work status, posting date, and technical skill indicators. Each table also contains a population-specific field describing its degree classification.

### `degree_jobs`

The `degree_jobs` table contains cleaned Data Analyst job postings identified as requiring a degree.

The table includes the `degree_type` field, which records the type of degree requirement identified during the data-cleaning process.

### `no_degree_jobs`

The `no_degree_jobs` table contains cleaned Data Analyst job postings identified as not requiring a degree.

The table includes the `no_degree_mention` Boolean field, which indicates whether the job posting was identified as a no-degree position.

### Primary Keys

Each table uses `job_id` as its primary key.

`job_id` uniquely identifies each job record within its respective table and was established during the data-cleaning process after duplicate records were removed.

The `job_id` values are independent between the two tables. They identify individual records within each population and do not represent a relationship between `degree_jobs` and `no_degree_jobs`.

Therefore, the two tables are designed as separate populations within the same relational database rather than as tables connected through a foreign-key relationship.

This structure allows SQL queries to analyze each population independently and compare the results across degree requirements.

## 5. Create SQLite Database

The cleaned Data Analyst job datasets will be stored in a SQLite3 relational database. SQLite3 was selected because it provides a lightweight relational database system that can be created and queried directly from Python without requiring a separate database server.

The database will contain two tables:

* `degree_jobs`
* `no_degree_jobs`

Each table will use `job_id` as its primary key. The tables will remain independent because their `job_id` values identify records within their respective populations and do not represent relationships between the two datasets.

The database will be created using Python's built-in `sqlite3` module, and the cleaned Pandas DataFrames will be loaded into the database for subsequent SQL analysis.

The SQLite database connection has been established. The two cleaned Pandas DataFrames will now be written to the database as separate relational tables.

Each DataFrame will become its corresponding table:

* `degree_jobs`
* `no_degree_jobs`

The `job_id` column will be retained as the unique record identifier for each table. The cleaned data will be loaded without replacing or modifying the values established during the data-cleaning notebooks.


In [11]:
import sqlite3
conn = sqlite3.connect("../data/data_analyst_jobs.db")
conn

In [14]:
degree_jobs.to_sql(
    "degree_jobs",
    conn,
    if_exists="replace",
    index=False
)

no_degree_jobs.to_sql(
    "no_degree_jobs",
    conn,
    if_exists="replace",
    index=False
)

18294

In [ ]:
# Verify tables
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,degree_jobs
1,no_degree_jobs


In [ ]:
# Verify row counts
pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*) FROM degree_jobs) AS degree_jobs_count,
        (SELECT COUNT(*) FROM no_degree_jobs) AS no_degree_jobs_count;
    """,
    conn
)

,degree_jobs_count,no_degree_jobs_count
0,759,18294


In [17]:
# Validate SQLite schema
pd.read_sql_query(
    "PRAGMA table_info(degree_jobs);",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,job_id,INTEGER,0,None,0
1,1,job_title,TEXT,0,None,0
2,2,salary_usd,INTEGER,0,None,0
3,3,employment_type,TEXT,0,None,0
4,4,company_location,TEXT,0,None,0
5,5,is_remote,INTEGER,0,None,0
6,6,employee_location,TEXT,0,None,0
7,7,job_skills,TEXT,0,None,0
8,8,degree_type,TEXT,0,None,0
9,9,posting_date,TEXT,0,None,0


In [18]:
# Validate SQLite schema
pd.read_sql_query(
    "PRAGMA table_info(no_degree_jobs);",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,job_id,INTEGER,0,None,0
1,1,job_title,TEXT,0,None,0
2,2,salary_usd,REAL,0,None,0
3,3,company_location,TEXT,0,None,0
4,4,is_remote,INTEGER,0,None,0
5,5,employee_location,TEXT,0,None,0
6,6,job_skills,TEXT,0,None,0
7,7,no_degree_mention,INTEGER,0,None,0
8,8,posting_date,TEXT,0,None,0
9,9,company_name,TEXT,0,None,0
